# Chapter 2: Pre-Model Workflow and Data Preprocessing

*Reproduction + theoretical summary of Chapter 2 of **scikit-learn Cookbook, Third Edition** (John Sukup, O'Reilly/Packt).*

The old saying *"garbage in, garbage out"* is the theme of this chapter: no algorithm can rescue data that is incomplete, badly scaled or encoded in a form the model cannot read. Preprocessing is also **not a one-off job**: because real data keeps changing, the same cleaning steps must be repeatable, which is why this chapter ends up leaning heavily on `Pipeline`.

The chapter has six recipes (plus a closing practical exercise). This notebook follows the same order:

| # | Recipe | Main tools |
|---|--------|-----------|
| 1 | The impact of raw data on model performance | what typically goes wrong |
| 2 | Handling missing data | `SimpleImputer`, `KNNImputer`, `IterativeImputer` |
| 3 | Scaling techniques | `StandardScaler`, `MinMaxScaler`, `Normalizer` (+ robust/power/quantile variants) |
| 4 | Encoding categorical variables | `OneHotEncoder`, `LabelEncoder`, `OrdinalEncoder`, `ColumnTransformer` |
| 5 | Pipelines | `Pipeline`, train/test discipline, leakage |
| 6 | Feature engineering | `PolynomialFeatures`, `KBinsDiscretizer`, `RFE`, `SelectFromModel` |
| 7 | Practical exercise | a full pipeline on California Housing |
| 8 | Decision guide | which scaler for which situation (the book's Figure 2.17) |

Each section has a short theory part, the book's code reproduced, and a few extra experiments that make the idea measurable. Where the book's wording is imprecise or out of date for current scikit-learn, a **📝 Note** explains the correction.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import sklearn

pd.set_option("display.precision", 3)
pd.set_option("display.width", 140)

print("scikit-learn:", sklearn.__version__)
print("NumPy       :", np.__version__)
print("pandas      :", pd.__version__)

scikit-learn: 1.9.1
NumPy       : 2.5.3
pandas      : 3.0.6


## 1. The impact of raw data on model performance

A model can only learn patterns that are actually present *and readable* in its input. Five problems show up again and again:

| Problem | What it does | Typical scikit-learn remedy |
|---------|--------------|-----------------------------|
| **Missing values** | most estimators refuse to run, or silently learn from a biased subset | `SimpleImputer`, `KNNImputer`, `IterativeImputer` |
| **Outliers** | distort means and standard deviations, pull fitted lines towards them | `RobustScaler`, careful inspection (they can also be the interesting cases!) |
| **Categorical variables** | text labels cannot be multiplied by weights | `OneHotEncoder`, `OrdinalEncoder` |
| **Different feature scales** | distance- and gradient-based models are dominated by the largest-scale feature | `StandardScaler`, `MinMaxScaler`, ... |
| **Data leakage** | information from outside the training set sneaks in, so scores look better than reality | split first, then preprocess inside a `Pipeline` |

Let's see the first two failure modes concretely, before fixing them in later sections.

In [2]:
from sklearn.linear_model import LogisticRegression

y_small = np.array([0, 0, 1, 1])

# (a) a missing value
X_nan = np.array([[1.0, 2.0], [2.0, np.nan], [3.0, 1.0], [4.0, 5.0]])
try:
    LogisticRegression().fit(X_nan, y_small)
except ValueError as e:
    print("Missing value  ->", type(e).__name__ + ":", str(e).splitlines()[0])

# (b) a text column
X_text = pd.DataFrame({"colour": ["red", "green", "blue", "red"]})
try:
    LogisticRegression().fit(X_text, y_small)
except ValueError as e:
    print("Text feature   ->", type(e).__name__ + ":", str(e).splitlines()[0])

Missing value  -> ValueError: Input X contains NaN.
Text feature   -> ValueError: could not convert string to float: 'red'


Both problems stop the model immediately, which is actually the *good* outcome: a loud error is better than a quietly wrong model.

**Outliers are subtler.** The next cell compares a z-score with a *robust* score on a tiny sample where the last value (200) is clearly abnormal. The robust version centres on the median and divides by the inter-quartile range (IQR), so the outlier cannot disturb its own yardstick:

$$z=\frac{x-\mu}{\sigma}\qquad\text{vs.}\qquad r=\frac{x-\text{median}}{\text{IQR}}$$

In [3]:
from sklearn.preprocessing import StandardScaler, RobustScaler

values = np.array([[10], [12], [11], [13], [12], [11], [10], [200.0]])

compare = pd.DataFrame({
    "value":  values.ravel(),
    "z-score": StandardScaler().fit_transform(values).ravel(),
    "robust":  RobustScaler().fit_transform(values).ravel(),
})
compare.round(2)

,value,z-score,robust
0,10.0,-0.40,-1.00
1,12.0,-0.37,0.33
2,11.0,-0.38,-0.33
3,13.0,-0.35,1.00
4,12.0,-0.37,0.33
5,11.0,-0.38,-0.33
6,10.0,-0.40,-1.00
7,200.0,2.65,125.67


The outlier gets a z-score of only **2.65**, below the popular "|z| > 3" cut-off, so a naive z-score rule would *not* flag it. This is called **masking**: the outlier inflates the standard deviation that is used to judge it. The robust score of the same point is **≈126**, which is impossible to miss, while the normal points stay in a tight band around 0.

> The book also reminds us that outliers are not always bad: in fraud or fault detection they are exactly the rare events we want to find.

## 2. Handling missing data

Most scikit-learn estimators raise an error when they meet `NaN`. The blunt fix is to drop incomplete rows, which is acceptable only when there is plenty of data and the loss is small. Otherwise we **impute**: fill each gap with an estimate computed from the available data. Three imputers are covered.

| Imputer | How the fill value is chosen | Cost |
|---------|-----------------------------|------|
| `SimpleImputer` | a per-column statistic: mean, median, most frequent, or a constant | very cheap |
| `KNNImputer` | looks at the *k* most similar rows and averages their value for that column | medium |
| `IterativeImputer` | models each incomplete column as a regression on the other columns, cycling round-robin | most expensive |

### 2.1 The book's toy dataset

20 rows × 10 uniform random features with roughly 20 % of the cells blanked out.

In [4]:
np.random.seed(2024)                      # same seed as the book
n_samples, n_features = 20, 10

data = {f"Feature{i+1}": np.random.uniform(0, 100, n_samples) for i in range(n_features)}
df = pd.DataFrame(data)

for column in df.columns:                 # blank out ~20% of each column
    mask = np.random.random(n_samples) < 0.2
    df.loc[mask, column] = np.nan

print("missing cells:", int(df.isna().sum().sum()), "of", df.size,
      f"({df.isna().mean().mean():.0%})")
df.head(6)

missing cells: 44 of 200 (22%)


,Feature1,Feature2,Feature3,Feature4,Feature5,Feature6,Feature7,Feature8,Feature9,Feature10
0,58.801,NaN,42.010,34.680,49.962,NaN,NaN,89.955,41.152,NaN
1,69.911,9.554,6.436,31.288,37.966,NaN,82.006,NaN,75.798,91.382
2,NaN,96.091,59.643,84.710,NaN,84.109,NaN,70.288,1.778,4.118
3,NaN,25.177,83.732,88.023,16.887,97.206,84.697,NaN,NaN,80.078
4,20.502,NaN,89.249,67.656,58.636,78.226,60.912,NaN,65.114,99.119
5,10.606,76.825,20.053,5.368,NaN,19.703,34.423,93.399,72.207,12.640


### 2.2 `SimpleImputer`

The strategy is chosen with `strategy=`: `"mean"`, `"median"`, `"most_frequent"` (the only one that also works for text columns) or `"constant"`.

In [5]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="mean")
imputed_df = pd.DataFrame(imputer.fit_transform(df), columns=df.columns)

print("mean of the observed Feature1 values:", round(df["Feature1"].mean(), 6))
imputed_df.head(6)

mean of the observed Feature1 values: 52.558605


,Feature1,Feature2,Feature3,Feature4,Feature5,Feature6,Feature7,Feature8,Feature9,Feature10
0,58.801,53.865,42.010,34.680,49.962,57.823,51.145,89.955,41.152,48.400
1,69.911,9.554,6.436,31.288,37.966,57.823,82.006,50.715,75.798,91.382
2,52.559,96.091,59.643,84.710,46.056,84.109,51.145,70.288,1.778,4.118
3,52.559,25.177,83.732,88.023,16.887,97.206,84.697,50.715,60.617,80.078
4,20.502,53.865,89.249,67.656,58.636,78.226,60.912,50.715,65.114,99.119
5,10.606,76.825,20.053,5.368,46.056,19.703,34.423,93.399,72.207,12.640


Row 2 of `Feature1` was `NaN`; it is now **52.558605**, exactly the column mean the book reports. Every gap in a column receives *the same* number, which is simple and robust but ignores any relationship between columns and shrinks the column's variance.

In [6]:
# the other strategies on the same column, looking only at the originally-missing rows
missing_rows = df["Feature1"].isna()
pd.DataFrame({
    "mean":          SimpleImputer(strategy="mean").fit_transform(df)[missing_rows, 0],
    "median":        SimpleImputer(strategy="median").fit_transform(df)[missing_rows, 0],
    "most_frequent": SimpleImputer(strategy="most_frequent").fit_transform(df)[missing_rows, 0],
    "constant(0)":   SimpleImputer(strategy="constant", fill_value=0).fit_transform(df)[missing_rows, 0],
}, index=df.index[missing_rows]).round(3)

,mean,median,most_frequent,constant(0)
2,52.559,60.245,1.911,0.0
3,52.559,60.245,1.911,0.0
7,52.559,60.245,1.911,0.0


(`most_frequent` is not meaningful for continuous random data, since almost every value is unique, so it falls back to the smallest value. It becomes useful for categorical columns.)

### 2.3 `KNNImputer`

For a row with a gap, the imputer finds the `n_neighbors` rows closest to it (using a Euclidean distance that simply skips missing coordinates, `nan_euclidean`) among rows that *do* have the missing column observed, and fills the gap with the **average of their values**.

> **📝 Note.** The book describes this as taking "the majority label" of the neighbours. That wording fits a *classifier*; the imputer actually averages the neighbours' values (a plain mean by default, or a distance-weighted mean with `weights="distance"`). The check below proves it.

In [7]:
from sklearn.impute import KNNImputer

knn_imputer = KNNImputer(n_neighbors=2)
knn_imputed_df = pd.DataFrame(knn_imputer.fit_transform(df), columns=df.columns)

# --- manual verification for one cell: row 2, Feature1 ---
from sklearn.metrics.pairwise import nan_euclidean_distances

row, col = 2, "Feature1"
candidates = df.index[df[col].notna()]                    # rows that could be donors
dist = nan_euclidean_distances(df.loc[[row]].values, df.loc[candidates].values).ravel()
nearest = candidates[np.argsort(dist)[:2]]

print("two nearest donor rows :", nearest.tolist())
print("their Feature1 values  :", df.loc[nearest, col].round(3).tolist())
print("average of the two     :", round(df.loc[nearest, col].mean(), 6))
print("KNNImputer's answer    :", round(knn_imputed_df.loc[row, col], 6))

two nearest donor rows : [12, 11]
their Feature1 values  : [60.245, 75.26]
average of the two     : 67.752344
KNNImputer's answer    : 67.752344


The manual average and the imputer output coincide, so `KNNImputer` really is "average of the k nearest rows".

### 2.4 `IterativeImputer`

Each column that has gaps takes a turn as the **target** $y$ of a regression whose inputs $X$ are the other columns; the fitted model predicts the gaps; the procedure repeats over all columns for several rounds until the estimates settle. It is still flagged *experimental*, so it needs an explicit enabling import.

In [8]:
from sklearn.experimental import enable_iterative_imputer   # noqa: F401  (must come first)
from sklearn.impute import IterativeImputer

iterative_imputer = IterativeImputer(random_state=0)
iterative_imputed_df = pd.DataFrame(iterative_imputer.fit_transform(df), columns=df.columns)
iterative_imputed_df.head(6)

,Feature1,Feature2,Feature3,Feature4,Feature5,Feature6,Feature7,Feature8,Feature9,Feature10
0,58.801,54.365,42.010,34.680,49.962,58.681,51.178,89.955,41.152,48.292
1,69.911,9.554,6.436,31.288,37.966,60.071,82.006,50.710,75.798,91.382
2,52.540,96.091,59.643,84.710,46.121,84.109,51.155,70.288,1.778,4.118
3,52.631,25.177,83.732,88.023,16.887,97.206,84.697,50.680,50.651,80.078
4,20.502,53.199,89.249,67.656,58.636,78.226,60.912,50.692,65.114,99.119
5,10.606,76.825,20.053,5.368,46.156,19.703,34.423,93.399,72.207,12.640


### 2.5 Which imputer is best? It depends on whether columns are related

Because we know nothing about the truth in the toy data, we cannot say which imputer is "better" there. So here is a small controlled experiment: take **complete** data, hide 15 % of the cells, impute, and measure the error against the hidden truth. We run it on two datasets:

* **correlated**: six features that move together (correlation 0.8),
* **independent**: six unrelated uniform features (like the book's toy data).

The error is reported *relative to the column standard deviation*, so 1.0 means "no better than guessing the average".

In [9]:
def relative_imputation_error(X_full, missing_rate=0.15, seed=0):
    rng = np.random.RandomState(seed)
    hidden = rng.rand(*X_full.shape) < missing_rate
    X_missing = X_full.copy()
    X_missing[hidden] = np.nan

    imputers = {
        "SimpleImputer (mean)":   SimpleImputer(strategy="mean"),
        "SimpleImputer (median)": SimpleImputer(strategy="median"),
        "KNNImputer (k=5)":       KNNImputer(n_neighbors=5),
        "IterativeImputer":       IterativeImputer(random_state=0),
    }
    scale = X_full.std()
    return {name: np.sqrt(np.mean((imp.fit_transform(X_missing)[hidden] - X_full[hidden]) ** 2)) / scale
            for name, imp in imputers.items()}

rng = np.random.RandomState(0)
cov = np.full((6, 6), 0.8) + 0.2 * np.eye(6)                    # correlation 0.8 between all pairs
X_correlated = rng.multivariate_normal(np.zeros(6), cov, size=500)
X_independent = rng.uniform(0, 100, size=(500, 6))

pd.DataFrame({
    "correlated features":  relative_imputation_error(X_correlated),
    "independent features": relative_imputation_error(X_independent),
}).round(3)

,correlated features,independent features
SimpleImputer (mean),0.961,0.986
SimpleImputer (median),0.962,0.989
KNNImputer (k=5),0.563,1.083
IterativeImputer,0.507,0.987


The result is a useful rule of thumb:

* When columns are **related**, `KNNImputer` and especially `IterativeImputer` cut the error by roughly 40-50 % compared with the mean, because they can "borrow" information from the other columns.
* When columns are **unrelated**, there is nothing to borrow, and the fancy methods are no better (KNN is even slightly worse) than the plain mean. The extra cost buys nothing.

The book's guideline on how much missing data to tolerate is similar in spirit: use the lightweight `SimpleImputer` when little is missing, move to KNN or iterative imputation for moderate amounts, and consider dropping a column that is mostly empty.

> **📝 Note.** The book's text says `SimpleImputer` is good "when >5 % of your dataset contains missing values". Given that the very next sentence treats the 5-10 % range, this is almost certainly a typo for **< 5 %**.

### 2.6 Some models do not need imputation at all

The book points out that tree-based models can cope with missing values. In current scikit-learn that is true for decision trees, random forests and histogram-based gradient boosting, but **not** for linear models:

In [10]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.datasets import make_classification

rng = np.random.RandomState(0)
X_c, y_c = make_classification(n_samples=300, n_features=6, random_state=0)
X_c[rng.rand(*X_c.shape) < 0.10] = np.nan                       # 10% missing

for model in [DecisionTreeClassifier(), RandomForestClassifier(n_estimators=20),
              HistGradientBoostingClassifier(), LogisticRegression()]:
    try:
        model.fit(X_c, y_c)
        print(f"{type(model).__name__:<32} fits data with NaN")
    except ValueError:
        print(f"{type(model).__name__:<32} refuses: needs imputation first")

DecisionTreeClassifier           fits data with NaN
RandomForestClassifier           fits data with NaN
HistGradientBoostingClassifier   fits data with NaN
LogisticRegression               refuses: needs imputation first


Two more practical details:

* `SimpleImputer(add_indicator=True)` appends one 0/1 column per incomplete feature that says *"this value was missing"*, which keeps the information that a value was absent (it can itself be predictive).
* Imputers are **transformers**: fit them on the training data only and reuse them on the test data (Section 5).

In [11]:
with_flag = SimpleImputer(strategy="median", add_indicator=True).fit_transform(df)
print("columns before:", df.shape[1], "-> after add_indicator:", with_flag.shape[1])

columns before: 10 -> after add_indicator: 20


## 3. Scaling techniques

Features rarely live on the same scale (age 0-100 versus income 0-100 000). Models that measure **distances** (k-NN, SVM, k-means) or are trained by **gradient descent** (linear/logistic regression, neural nets) are then dominated by the large-scale features. Scaling levels the field.

Two words that are often confused:

* **Standardization** shifts and rescales each *column* to mean 0 and standard deviation 1.
* **Min-max scaling** squeezes each *column* into a fixed interval, usually [0, 1]. (The book calls this "normalization" in the intro of the recipe.)
* `Normalizer` is something else: it rescales each **row** (sample) to unit length.

| Scaler | Works on | Formula | Result |
|--------|----------|---------|--------|
| `StandardScaler` | columns | $z=\dfrac{x-\mu}{\sigma}$ | mean 0, std 1 (values can be negative) |
| `MinMaxScaler` | columns | $x'=\dfrac{x-x_{\min}}{x_{\max}-x_{\min}}$ | every column in [0, 1] |
| `Normalizer` | **rows** | $\mathbf{x}'=\dfrac{\mathbf{x}}{\lVert\mathbf{x}\rVert}$, with $\lVert\mathbf{x}\rVert_1=\sum_j\lvert x_j\rvert$ or $\lVert\mathbf{x}\rVert_2=\sqrt{\sum_j x_j^2}$ | every row has norm 1 |

The book uses the imputed DataFrame from Section 2.4 as input, so we do the same.

In [12]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler, Normalizer

standard_df = pd.DataFrame(StandardScaler().fit_transform(iterative_imputed_df),
                           columns=iterative_imputed_df.columns)
minmax_df = pd.DataFrame(MinMaxScaler().fit_transform(iterative_imputed_df),
                         columns=iterative_imputed_df.columns)
normalized_df = pd.DataFrame(Normalizer().fit_transform(iterative_imputed_df),     # default norm = "l2"
                             columns=iterative_imputed_df.columns)

def summarise(frame):
    # one-number summaries across all 10 columns
    return pd.Series({"avg column mean": frame.mean().mean(),
                      "avg column std":  frame.std(ddof=0).mean(),
                      "overall min":     frame.values.min(),
                      "overall max":     frame.values.max()})

pd.DataFrame({
    "original (imputed)": summarise(iterative_imputed_df),
    "StandardScaler":     summarise(standard_df),
    "MinMaxScaler":       summarise(minmax_df),
    "Normalizer (L2)":    summarise(normalized_df),
}).round(3)

,original (imputed),StandardScaler,MinMaxScaler,Normalizer (L2)
avg column mean,52.853,-0.000,0.525,0.289
avg column std,24.400,1.000,0.276,0.126
overall min,0.878,-2.410,0.000,0.005
overall max,99.974,2.024,1.000,0.600


Reading the table: after `StandardScaler` the average column mean is 0 and std is 1; after `MinMaxScaler` the overall range is exactly [0, 1]; `Normalizer` produces small values because it works along rows. Let's confirm what `Normalizer` really guarantees:

In [13]:
l2 = Normalizer(norm="l2").fit_transform(iterative_imputed_df)
l1 = Normalizer(norm="l1").fit_transform(iterative_imputed_df)

print("L2 row lengths (first 5):", np.linalg.norm(l2, axis=1)[:5].round(6))
print("L1 row sums    (first 5):", np.abs(l1).sum(axis=1)[:5].round(6))

# tiny hand-checkable example: the row [3, 4] has L2 length 5 -> [0.6, 0.8]
print("\n[3, 4] ->", Normalizer().fit_transform([[3.0, 4.0]]).ravel(), "(L2)")
print("[3, 4] ->", Normalizer(norm="l1").fit_transform([[3.0, 4.0]]).ravel().round(4), "(L1)")

L2 row lengths (first 5): [1. 1. 1. 1. 1.]
L1 row sums    (first 5): [1. 1. 1. 1. 1.]

[3, 4] -> [0.6 0.8] (L2)
[3, 4] -> [0.4286 0.5714] (L1)


Every row now has length exactly 1 (L2) or absolute sum 1 (L1). `Normalizer` keeps the *direction* of each sample and discards its magnitude, so it is handy for text/sparse data or cosine-similarity style comparisons, and rarely what you want for ordinary tabular columns.

### 3.1 Beyond the three: scalers that appear in the book's decision guide

The decision chart at the end of the chapter (Section 8) mentions more transformers. A quick look on a **skewed** feature (log-normal, with one extreme value) shows what each one does to the shape of the data:

In [14]:
from scipy.stats import skew
from sklearn.preprocessing import MaxAbsScaler, PowerTransformer, QuantileTransformer

rng = np.random.RandomState(0)
x = rng.lognormal(mean=0.0, sigma=1.0, size=(500, 1))
x[0] = 200.0                                            # one extreme value

transformers = {
    "raw (no scaling)":              None,
    "StandardScaler":                StandardScaler(),
    "MinMaxScaler":                  MinMaxScaler(),
    "RobustScaler":                  RobustScaler(),
    "MaxAbsScaler":                  MaxAbsScaler(),
    "PowerTransformer (Yeo-Johnson)": PowerTransformer(),
    "QuantileTransformer (normal)":  QuantileTransformer(n_quantiles=200, output_distribution="normal", random_state=0),
}

rows = {}
for name, t in transformers.items():
    z = x if t is None else t.fit_transform(x)
    rows[name] = {"skewness": skew(z.ravel()), "min": z.min(), "median": np.median(z), "max": z.max()}

pd.DataFrame(rows).T.round(2)

,skewness,min,median,max
raw (no scaling),20.90,0.06,0.95,200.00
StandardScaler,20.90,-0.21,-0.11,21.86
MinMaxScaler,20.90,0.00,0.00,1.00
RobustScaler,20.90,-0.61,0.00,137.41
MaxAbsScaler,20.90,0.00,0.00,1.00
PowerTransformer (Yeo-Johnson),0.15,-2.02,-0.05,2.58
QuantileTransformer (normal),-0.02,-5.20,-0.00,5.20


What the numbers tell us:

* `StandardScaler`, `MinMaxScaler`, `RobustScaler` and `MaxAbsScaler` are all **linear** rescalings, so the **skewness is unchanged** (≈ 21 here, caused by the extreme value). They only move the data and change its units.
* `MinMaxScaler` is the most hurt by the outlier: that single point defines the maximum, so almost every other value is squashed close to 0.
* `RobustScaler` uses the median and IQR, so the bulk of the data keeps a sensible spread (median 0).
* `PowerTransformer` and `QuantileTransformer` are **non-linear**: they actually reshape the distribution towards a bell curve, so the skewness drops sharply.

### 3.2 Does scaling really matter? It depends on the model

The book says scaling is vital for distance- and gradient-based models and unnecessary for trees. We can measure that on the Wine dataset (13 chemical features on very different scales), using 5-fold cross-validated accuracy. The scaler sits **inside a pipeline** so that it is re-fitted on each training fold.

In [15]:
from sklearn.datasets import load_wine
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

wine = load_wine()
X_wine, y_wine = wine.data, wine.target
print("feature ranges (min -> max) for a few of the 13 columns:")
for j in [0, 2, 12]:
    print(f"  {wine.feature_names[j]:<12}: {X_wine[:, j].min():8.2f} -> {X_wine[:, j].max():8.2f}")

models = {
    "k-NN":                KNeighborsClassifier(),
    "SVM (RBF)":           SVC(),
    "Logistic regression": LogisticRegression(max_iter=5000),
    "Random forest":       RandomForestClassifier(random_state=0),
}
scalers = {"no scaling": None, "StandardScaler": StandardScaler(),
           "MinMaxScaler": MinMaxScaler(), "RobustScaler": RobustScaler()}

results = pd.DataFrame(index=models.keys(), columns=scalers.keys(), dtype=float)
for m_name, model in models.items():
    for s_name, scaler in scalers.items():
        pipe = make_pipeline(scaler, model) if scaler is not None else model
        results.loc[m_name, s_name] = cross_val_score(pipe, X_wine, y_wine, cv=5).mean()

results.round(3)

feature ranges (min -> max) for a few of the 13 columns:
  alcohol     :    11.03 ->    14.83
  ash         :     1.36 ->     3.23
  proline     :   278.00 ->  1680.00


,no scaling,StandardScaler,MinMaxScaler,RobustScaler
k-NN,0.691,0.949,0.950,0.938
SVM (RBF),0.663,0.983,0.978,0.983
Logistic regression,0.961,0.983,0.978,0.989
Random forest,0.983,0.983,0.983,0.983


The table is the whole argument for scaling in one picture:

* **k-NN** jumps from about 0.69 to about 0.95 and **SVM** from about 0.66 to about 0.98, since without scaling the large-valued column (`proline`, which runs from hundreds into the thousands, versus values of order 1-10 elsewhere) decides every distance.
* **Logistic regression** improves a little (and converges more easily).
* **Random forest** scores the same under every scaler. A tree only asks "is this value above a threshold?", and a monotonic rescaling does not change the order of values, so splits are unaffected.

> **📝 Note.** The book says scaling would have a "detrimental impact" on tree models. In practice it is simply **unnecessary** (identical scores above), not harmful. The conclusion is the same: skip it for trees, because it adds code and no benefit.

## 4. Encoding categorical variables

Models need numbers, so text categories have to be converted. First decide which kind of variable you have:

* **Nominal**: categories without order (colour, department, city).
* **Ordinal**: categories with a meaningful order (Junior < Senior < Manager; rating 1-5).

| Encoder | Output | Use for |
|---------|--------|---------|
| `OneHotEncoder` | one 0/1 column per category | nominal features |
| `OrdinalEncoder` | one integer column, **order you specify** | ordinal features |
| `LabelEncoder` | integers `0..K-1` | the **target** `y` of a classification problem |
| `ColumnTransformer` | (a container) | applying *different* preprocessing to different columns |

### 4.1 The book's toy categorical dataset

In [16]:
np.random.seed(2024)
categorical_data = pd.DataFrame({
    "Department": np.random.choice(["A", "B", "C", "D"], size=20),
    "Position":   np.random.choice(["Junior", "Senior", "Manager"], size=20),
    "Location":   np.random.choice(["NY", "SF", "LA", "CHI"], size=20),
})
categorical_data.head(6)

,Department,Position,Location
0,A,Manager,LA
1,C,Manager,LA
2,A,Junior,NY
3,A,Manager,LA
4,D,Manager,NY
5,A,Senior,LA


### 4.2 `OneHotEncoder`

A feature with $K$ categories becomes $K$ binary columns, exactly one of which is 1 in each row. No order is implied, so the model treats each category independently. The price is width: 3 features with 4, 3 and 4 categories turn into 11 columns, and high-cardinality features can explode into very wide, mostly-zero matrices.

In [17]:
from sklearn.preprocessing import OneHotEncoder

onehot_encoder = OneHotEncoder(sparse_output=False)
onehot_encoded_df = pd.DataFrame(onehot_encoder.fit_transform(categorical_data),
                                 columns=onehot_encoder.get_feature_names_out())

print("shape:", categorical_data.shape, "->", onehot_encoded_df.shape)
onehot_encoded_df.head(6)

shape: (20, 3) -> (20, 11)


,Department_A,Department_B,Department_C,Department_D,Position_Junior,Position_Manager,Position_Senior,Location_CHI,Location_LA,Location_NY,Location_SF
0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
1,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
2,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
3,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
4,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
5,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0


Useful options (not shown in the book):

* `handle_unknown="ignore"`: a category first seen at prediction time becomes all-zeros instead of raising an error. This matters in production.
* `drop="first"`: drops one column per feature to avoid perfect redundancy (the *dummy-variable trap*), which can matter for unregularised linear models.

In [18]:
loc = categorical_data[["Location"]]

safe = OneHotEncoder(handle_unknown="ignore", sparse_output=False).fit(loc)
print("categories learned      :", safe.categories_[0].tolist())
print("unseen value 'XX' ->    :", safe.transform(pd.DataFrame({"Location": ["XX"]})).ravel(), "(all zeros, no crash)")

strict = OneHotEncoder(sparse_output=False).fit(loc)           # default: handle_unknown='error'
try:
    strict.transform(pd.DataFrame({"Location": ["XX"]}))
except ValueError as e:
    print("default encoder        :", str(e).splitlines()[0])

dropped = OneHotEncoder(drop="first", sparse_output=False).fit(categorical_data)
print("\nwith drop='first', width:", dropped.transform(categorical_data).shape[1], "(instead of 11)")

categories learned      : ['CHI', 'LA', 'NY', 'SF']
unseen value 'XX' ->    : [0. 0. 0. 0.] (all zeros, no crash)
default encoder        : Found unknown categories ['XX'] in column 0 during transform

with drop='first', width: 8 (instead of 11)


### 4.3 `LabelEncoder`, and why it is risky for features

`LabelEncoder` maps every category to an integer. Done per column (as in the book) it is compact, but the integers **suggest an order and distances that do not exist**: Is "SF"(3) really "three times" "CHI"(0)? Linear and distance-based models will act as if so.

In [19]:
from sklearn.preprocessing import LabelEncoder

label_encoded_df = pd.DataFrame()
for column in categorical_data.columns:
    le = LabelEncoder()                                    # a fresh encoder per column
    label_encoded_df[f"{column}_encoded"] = le.fit_transform(categorical_data[column])
    print(f"{column:<10} classes (alphabetical -> 0,1,2...):", le.classes_.tolist())

label_encoded_df.head(6)

Department classes (alphabetical -> 0,1,2...): ['A', 'B', 'C', 'D']
Position   classes (alphabetical -> 0,1,2...): ['Junior', 'Manager', 'Senior']
Location   classes (alphabetical -> 0,1,2...): ['CHI', 'LA', 'NY', 'SF']


,Department_encoded,Position_encoded,Location_encoded
0,0,1,1
1,2,1,1
2,0,0,2
3,0,1,1
4,3,1,2
5,0,2,1


> **📝 Note.** The integers are assigned **alphabetically**, not by meaning: here `Junior=0, Manager=1, Senior=2`, which even puts *Manager* below *Senior*. Per scikit-learn's documentation `LabelEncoder` is designed for the **target** variable. For *input features* the right tool is `OrdinalEncoder`, where you state the order yourself:

In [20]:
from sklearn.preprocessing import OrdinalEncoder

order = [["Junior", "Senior", "Manager"]]                   # explicit, meaningful order
ordinal = OrdinalEncoder(categories=order)
position_codes = ordinal.fit_transform(categorical_data[["Position"]]).ravel()

pd.DataFrame({"Position": categorical_data["Position"], "label-encoded": label_encoded_df["Position_encoded"],
              "ordinal (Junior<Senior<Manager)": position_codes}).drop_duplicates().sort_values("ordinal (Junior<Senior<Manager)")

,Position,label-encoded,ordinal (Junior<Senior<Manager)
2,Junior,0,0.0
5,Senior,2,1.0
0,Manager,1,2.0


With `OrdinalEncoder` the codes now respect the real ranking (Junior 0 < Senior 1 < Manager 2). Use it **only** for genuinely ordinal features; for nominal ones prefer one-hot.

### 4.4 `ColumnTransformer`: different treatment per column

Real tables mix numbers and categories. `ColumnTransformer` takes a list of `(name, transformer, columns)` triples and glues the outputs together side by side. Columns not mentioned are dropped unless `remainder="passthrough"` keeps them untouched.

In [21]:
from sklearn.compose import ColumnTransformer

np.random.seed(2024)
mixed_data = pd.DataFrame({
    "Age":        np.random.randint(25, 65, size=20),
    "Salary":     np.round(np.random.normal(60000, 15000, size=20), 2),
    "Experience": np.random.randint(1, 20, size=20),
    "Department": np.random.choice(["IT", "HR", "Sales", "Finance"], size=20),
    "Position":   np.random.choice(["Junior", "Senior", "Manager"], size=20),
})
mixed_data.head(6)

,Age,Salary,Experience,Department,Position
0,33,59420.36,12,Finance,Manager
1,57,82895.92,17,Sales,Junior
2,25,38165.76,16,Finance,Manager
3,52,38242.36,7,IT,Junior
4,61,55088.65,8,Sales,Manager
5,26,78688.88,9,Sales,Senior


In [22]:
column_transformer = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(),  ["Age", "Salary", "Experience"]),
        ("cat", OneHotEncoder(sparse_output=False), ["Department", "Position"]),
    ],
    remainder="passthrough",
)

# set_output("pandas") makes the result a DataFrame with proper column names
# (the book builds the names by hand).
column_transformer.set_output(transform="pandas")
transformed_df = column_transformer.fit_transform(mixed_data)

print("shape:", mixed_data.shape, "->", transformed_df.shape)
transformed_df.head(6)

shape: (20, 5) -> (20, 10)


,num__Age,num__Salary,num__Experience,cat__Department_Finance,cat__Department_HR,cat__Department_IT,cat__Department_Sales,cat__Position_Junior,cat__Position_Manager,cat__Position_Senior
0,-1.045,-0.305,0.327,1.0,0.0,0.0,0.0,0.0,1.0,0.0
1,0.728,1.105,1.262,0.0,0.0,0.0,1.0,1.0,0.0,0.0
2,-1.636,-1.582,1.075,1.0,0.0,0.0,0.0,0.0,1.0,0.0
3,0.358,-1.577,-0.608,0.0,0.0,1.0,0.0,1.0,0.0,0.0
4,1.023,-0.565,-0.421,0.0,0.0,0.0,1.0,0.0,1.0,0.0
5,-1.562,0.852,-0.234,0.0,0.0,0.0,1.0,0.0,0.0,1.0


The 3 numeric columns were standardised and the 2 categorical columns became 4 + 3 = 7 indicator columns (total 10). The `num__` / `cat__` prefixes come from the step names we gave, which keeps the origin of every output column traceable.

## 5. Introduction to pipelines

A `Pipeline` is an ordered list of `(name, step)` pairs: zero or more **transformers**, optionally followed by one final **estimator**. The book lists five benefits:

1. **Sequential execution**: the same steps, in the same order, for train and test data.
2. **Simpler code**: one object instead of many loose variables.
3. **Consistency**: fewer chances for leakage (below).
4. **Easier tuning**: search over preprocessing *and* model settings together.
5. **Modularity**: swap a step without rewriting the rest.

### 5.1 Split **before** you transform

Always call `train_test_split` first. Any statistic computed from the whole dataset (a mean, a scaling range, even *which features look useful*) would otherwise carry information about the test rows into training, giving **over-optimistic** scores. After fitting the pipeline on the training part, use `transform()` / `predict()` on the test part, **never** `fit()`.

We reproduce the book's imputer → scaler pipeline on the toy data with gaps from Section 2.

In [23]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split

X_train, X_test = train_test_split(df, test_size=0.2, random_state=2024)      # split FIRST
print("train rows:", len(X_train), " test rows:", len(X_test))

pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="mean")),      # step 1: fill the gaps
    ("scaler",  StandardScaler()),                    # step 2: standardise
])

X_train_t = pd.DataFrame(pipeline.fit_transform(X_train), columns=df.columns, index=X_train.index)
X_test_t  = pd.DataFrame(pipeline.transform(X_test),      columns=df.columns, index=X_test.index)

print("\ntrain: column means after the pipeline (≈ 0 by construction):")
print(X_train_t.mean().round(3).values)
print("\ntest : column means after the pipeline (NOT 0: it uses the *training* statistics):")
print(X_test_t.mean().round(3).values)

train rows: 16  test rows: 4

train: column means after the pipeline (≈ 0 by construction):
[-0. -0.  0.  0.  0.  0. -0.  0. -0.  0.]

test : column means after the pipeline (NOT 0: it uses the *training* statistics):
[ 1.009 -0.15  -0.023  0.907 -0.495  0.017 -0.317 -0.864  0.163  0.274]


The test means are not zero, and that is **correct**: the test rows are measured with the training ruler, exactly as they would be for brand-new data in production. (If we had re-fitted on the test set, the means would have been forced to zero and any distribution shift would have been hidden. See the Chapter 1 notebook, Section 3.2.)

### 5.2 Visualising a pipeline

`set_config(display="diagram")` makes notebooks draw the pipeline as an interactive block diagram (click a block to expand its parameters). Plain `print(pipeline)` gives the text form if your viewer cannot render HTML.

In [24]:
from sklearn import set_config
set_config(display="diagram")

pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('imputer', ...), ('scaler', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](10,)","['Feature1','Feature2','Feature3',...,'Feature8','Feature9','Feature10']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,10
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'mean'
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. For string or object data types,`fill_value` must be a string.If `None`, `fill_value` will be 0 when imputing numericaldata and ""missing_value"" for strings or object data types.",Non

### 5.3 What leakage looks like in numbers

The classic demonstration: build a dataset of **pure noise** (2 000 random features, random 0/1 labels), where *no model can truly do better than 50 %*. Then select the "best" 20 features either **before** cross-validation (leaky) or **inside** the pipeline (correct).

In [25]:
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import RepeatedStratifiedKFold

rng = np.random.RandomState(1)
X_noise = rng.normal(size=(100, 2000))
y_noise = rng.randint(0, 2, size=100)                 # labels unrelated to X
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=0)

# WRONG: feature selection sees ALL labels, then we cross-validate
X_leaky = SelectKBest(f_classif, k=20).fit_transform(X_noise, y_noise)
leaky = cross_val_score(LogisticRegression(max_iter=1000), X_leaky, y_noise, cv=cv).mean()

# RIGHT: selection is a pipeline step, re-fitted on each training fold only
safe_pipe = make_pipeline(SelectKBest(f_classif, k=20), LogisticRegression(max_iter=1000))
safe = cross_val_score(safe_pipe, X_noise, y_noise, cv=cv).mean()

print(f"selection BEFORE cross-validation (leaky): accuracy = {leaky:.3f}")
print(f"selection INSIDE the pipeline (correct)  : accuracy = {safe:.3f}")
print("true achievable accuracy on pure noise   : 0.500")

selection BEFORE cross-validation (leaky): accuracy = 0.842
selection INSIDE the pipeline (correct)  : accuracy = 0.480
true achievable accuracy on pure noise   : 0.500


The leaky version "discovers" a model with roughly 84 % accuracy **on data that contains no signal at all**. It is not learning anything; the selection step already peeked at the labels of the rows it was later tested on. The correct pipeline hovers around chance, which is the honest answer. This is the strongest argument for putting *every* data-dependent step inside the pipeline.

### 5.4 A realistic end-to-end pipeline

A more complete example than the book's: a synthetic employee table with **missing values in both numeric and categorical columns**, a `ColumnTransformer` that treats each type appropriately, and a final regressor. The target is salary.

In [26]:
rng = np.random.RandomState(42)
n = 400
emp = pd.DataFrame({
    "Age":        rng.randint(22, 62, n).astype(float),
    "Experience": rng.randint(0, 30, n).astype(float),
    "Department": rng.choice(["IT", "HR", "Sales", "Finance"], n).astype(object),
    "Position":   rng.choice(["Junior", "Senior", "Manager"], n, p=[0.5, 0.3, 0.2]).astype(object),
})
emp["Experience"] = np.minimum(emp["Experience"], emp["Age"] - 21)        # can't have more experience than working years

salary = (30000 + 1200 * emp["Experience"]
          + emp["Position"].map({"Junior": 0, "Senior": 15000, "Manager": 35000})
          + emp["Department"].map({"IT": 10000, "HR": 0, "Sales": 5000, "Finance": 12000})
          + rng.normal(0, 6000, n))

for col in ["Age", "Experience", "Department", "Position"]:               # ~7% missing everywhere
    emp.loc[rng.rand(n) < 0.07, col] = np.nan

print("missing values per column:", emp.isna().sum().to_dict())

emp_train, emp_test, sal_train, sal_test = train_test_split(emp, salary, test_size=0.25, random_state=0)

missing values per column: {'Age': 29, 'Experience': 30, 'Department': 32, 'Position': 25}


In [27]:
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV

num_cols = ["Age", "Experience"]
cat_cols = ["Department", "Position"]

preprocess = ColumnTransformer([
    ("num", Pipeline([("imputer", SimpleImputer(strategy="median")),
                      ("scaler",  StandardScaler())]), num_cols),
    ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                      ("onehot",  OneHotEncoder(handle_unknown="ignore"))]), cat_cols),
])

model_pipe = Pipeline([("prep", preprocess), ("model", Ridge())])
model_pipe

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('prep', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output 

In [28]:
# tune preprocessing AND model together; note the  <step>__<sub-step>__<param>  naming
param_grid = {
    "prep__num__imputer__strategy": ["mean", "median"],
    "model__alpha": [0.1, 1, 10, 100],
}
search = GridSearchCV(model_pipe, param_grid, cv=5, scoring="r2")
search.fit(emp_train, sal_train)

print("best settings :", search.best_params_)
print(f"CV R^2        : {search.best_score_:.3f}")
print(f"test R^2      : {search.score(emp_test, sal_test):.3f}")

names = search.best_estimator_.named_steps["prep"].get_feature_names_out()
print("\nfeatures seen by the model:", names.tolist())

best settings : {'model__alpha': 1, 'prep__num__imputer__strategy': 'mean'}
CV R^2        : 0.745
test R^2      : 0.782

features seen by the model: ['num__Age', 'num__Experience', 'cat__Department_Finance', 'cat__Department_HR', 'cat__Department_IT', 'cat__Department_Sales', 'cat__Position_Junior', 'cat__Position_Manager', 'cat__Position_Senior']


Raw data with gaps in four columns goes in, and a tuned model comes out of **one** object: a single `fit()` imputes, scales, one-hot encodes and regresses, a single `predict()` repeats the same steps on new rows, and the grid search was free to tune the imputation strategy alongside the regularisation strength. Without a pipeline each of these would be a separate, easy-to-get-wrong manual step.

> The same object can be saved with `joblib.dump(search.best_estimator_, "model.joblib")` for deployment (see the Chapter 1 notebook, Section 5.2).

## 6. Feature engineering

*Feature engineering* is an umbrella term for two activities:

1. **Feature extraction / creation**: derive *new* columns from existing ones (e.g. total spend = price × quantity).
2. **Feature selection**: keep only the informative columns and drop the rest.

Good features can make a simple model work well, while a long list of noisy ones invites overfitting and slows everything down. Selection methods are traditionally grouped into **filter** methods (judge each feature by a statistic), **wrapper** methods (train models on candidate subsets and compare) and hybrids/embedded methods. The book demonstrates two creation tools and two selection tools.

### 6.1 `PolynomialFeatures`

For input columns $a,b,c$ and `degree=2` it creates the bias term, the originals, the **squares** and all **pairwise products** (interactions):

$$1,\ a,\ b,\ c,\ a^2,\ ab,\ ac,\ b^2,\ bc,\ c^2$$

This lets a *linear* model fit curved relationships and interactions.

In [29]:
from sklearn.preprocessing import PolynomialFeatures

rng = np.random.RandomState(0)
X_small = pd.DataFrame(rng.normal(size=(5, 3)), columns=["a", "b", "c"])

poly = PolynomialFeatures(degree=2)
poly_df = pd.DataFrame(poly.fit_transform(X_small), columns=poly.get_feature_names_out(X_small.columns))

print("3 input columns ->", poly_df.shape[1], "output columns")
print("only interactions (no squares, no bias):",
      PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)
      .fit(X_small).get_feature_names_out().tolist())
poly_df.round(2)

3 input columns -> 10 output columns
only interactions (no squares, no bias): ['a', 'b', 'c', 'a b', 'a c', 'b c']


,1,a,b,c,a^2,a b,a c,b^2,b c,c^2
0,1.0,1.76,0.40,0.98,3.11,0.71,1.73,0.16,0.39,0.96
1,1.0,2.24,1.87,-0.98,5.02,4.18,-2.19,3.49,-1.83,0.96
2,1.0,0.95,-0.15,-0.10,0.90,-0.14,-0.10,0.02,0.02,0.01
3,1.0,0.41,0.14,1.45,0.17,0.06,0.60,0.02,0.21,2.11
4,1.0,0.76,0.12,0.44,0.58,0.09,0.34,0.01,0.05,0.20


The number of output columns grows quickly with the degree (for $p$ inputs and degree $d$ it is $\binom{p+d}{d}$), so keep degrees low and pair it with feature selection or regularisation.

**Does it help?** A straight line cannot fit a parabola, but a linear model on polynomial features can:

In [30]:
from sklearn.linear_model import LinearRegression

rng = np.random.RandomState(3)
x = rng.uniform(-3, 3, size=(200, 1))
y_curve = x.ravel() ** 2 + rng.normal(0, 0.5, 200)                      # y = x^2 + noise
x_tr, x_te, y_tr, y_te = train_test_split(x, y_curve, test_size=0.3, random_state=0)

plain = LinearRegression().fit(x_tr, y_tr)
curved = make_pipeline(PolynomialFeatures(degree=2, include_bias=False), LinearRegression()).fit(x_tr, y_tr)

print(f"straight line         test R^2 = {plain.score(x_te, y_te):.3f}")
print(f"+ PolynomialFeatures  test R^2 = {curved.score(x_te, y_te):.3f}")

straight line         test R^2 = -0.012
+ PolynomialFeatures  test R^2 = 0.942


The straight line explains almost none of the variance (R² ≈ 0), while the same linear regression with the squared term explains nearly all of it. Nothing about the *model* changed, only the *features*. That is feature engineering in a nutshell.

### 6.2 `KBinsDiscretizer`

Turns a continuous column into a handful of **buckets**. `n_bins` sets how many; `strategy` sets where the boundaries go:

| strategy | boundaries | each bin has... |
|----------|-----------|-----------------|
| `uniform` | equal-width intervals | equal *width* (counts may be very unequal) |
| `quantile` | at quantiles | roughly equal *number of samples* |
| `kmeans` | centred on 1-D k-means clusters | natural groupings |

`encode="ordinal"` returns the bin number; `encode="onehot"` returns indicator columns.

The toy column below has a long right tail, which makes the strategies easy to tell apart:

In [31]:
from sklearn.preprocessing import KBinsDiscretizer

v = np.array([[1], [2], [2], [3], [3], [3], [4], [10], [11], [50.0]])

rows = {}
for strategy in ["uniform", "quantile", "kmeans"]:
    kb = KBinsDiscretizer(n_bins=3, encode="ordinal", strategy=strategy).fit(v)
    rows[strategy] = {"bin edges": kb.bin_edges_[0].round(1).tolist(),
                      "bin of each value": kb.transform(v).ravel().astype(int).tolist(),
                      "samples per bin": np.bincount(kb.transform(v).ravel().astype(int), minlength=3).tolist()}
pd.DataFrame(rows).T

,bin edges,bin of each value,samples per bin
uniform,"[1.0, 17.3, 33.7, 50.0]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 2]","[9, 0, 1]"
quantile,"[1.0, 3.0, 4.0, 50.0]","[0, 0, 0, 1, 1, 1, 2, 2, 2, 2]","[3, 3, 4]"
kmeans,"[1.0, 6.5, 30.2, 50.0]","[0, 0, 0, 0, 0, 0, 0, 1, 1, 2]","[7, 2, 1]"


With `uniform`, nine of the ten values land in the first bin because the single large value stretches the range. `quantile` spreads the samples roughly evenly (3-4 per bin) and `kmeans` puts the isolated values in their own bins. Binning discards fine detail, but it can make non-linear effects usable by a linear model and tames outliers.

### 6.3 Feature selection: `RFE`

**Recursive Feature Elimination** repeatedly (1) trains the estimator on the current features, (2) removes the least important one(s), and (3) repeats until the requested number remains. The estimator must expose `coef_` or `feature_importances_`. Its `ranking_` attribute gives 1 to the surviving/most valuable feature and larger numbers to features that were eliminated earlier. `RFECV` adds cross-validation to *choose* how many features to keep.

The book applies it to an arbitrary table. Here we use a regression problem where we **know the truth**: 9 features, of which only the first 3 (`f0, f1, f2`) actually influence the target.

In [32]:
from sklearn.datasets import make_regression
from sklearn.feature_selection import RFE, RFECV, SelectFromModel

X_r, y_r = make_regression(n_samples=200, n_features=9, n_informative=3, noise=5,
                           shuffle=False, random_state=0)                 # informative = first 3 columns
cols = [f"f{i}" for i in range(9)]
X_r = pd.DataFrame(StandardScaler().fit_transform(X_r), columns=cols)     # standardised -> coefficients comparable

rfe = RFE(estimator=LinearRegression(), n_features_to_select=1).fit(X_r, y_r)
ranking = pd.Series(rfe.ranking_, index=cols).sort_values()
print("RFE ranking (1 = best):")
print(ranking.to_string())

rfecv = RFECV(LinearRegression(), cv=5).fit(X_r, y_r)
print("\nRFECV chose", rfecv.n_features_, "features:", np.array(cols)[rfecv.support_].tolist())

RFE ranking (1 = best):
f1    1
f0    2
f2    3
f4    4
f5    5
f8    6
f7    7
f6    8
f3    9

RFECV chose 3 features: ['f0', 'f1', 'f2']


RFE ranks the three truly informative columns (`f1`, `f0`, `f2`) at positions 1-3 and pushes the six noise columns to the bottom, so it recovered the real structure. `RFECV`, which also decides *how many* to keep, independently settles on exactly those three.

### 6.4 Feature selection: `SelectFromModel`

Trains the estimator **once**, reads its importance scores (`coef_` or `feature_importances_`), and keeps the features whose importance is above a `threshold` (`"mean"`, `"median"`, or a number; for linear models the magnitude $|w_j|$ is used). It is cheaper than RFE, because there is no repeated retraining.

In [33]:
selector = SelectFromModel(estimator=LinearRegression(), prefit=False, threshold="mean")
selector.fit(X_r, y_r)

feature_importance = pd.DataFrame({
    "Feature":    cols,
    "Importance": selector.estimator_.coef_,
    "Selected":   selector.get_support(),
}).sort_values("Importance", key=abs, ascending=False)

print("threshold (mean of |coef|):", round(selector.threshold_, 3))
feature_importance.round(3)

threshold (mean of |coef|): 14.63


,Feature,Importance,Selected
1,f1,86.885,True
0,f0,35.541,True
2,f2,7.949,False
4,f4,0.428,False
5,f5,0.326,False
8,f8,0.182,False
7,f7,0.138,False
6,f6,0.123,False
3,f3,-0.095,False


The threshold is the mean absolute coefficient (≈ 15); only `f1` and `f0` exceed it, so only they are kept. This shows the trade-off: `SelectFromModel` is fast but its cut-off is blunt: `f2` is informative, yet its coefficient (≈ 8) sits below the threshold (≈ 15), so it is dropped. Lowering the threshold (`threshold="median"` or a number) would keep it. RFE/RFECV is slower but chose all three.

> **Important:** coefficients are only comparable when the features share a scale. That is why the data was standardised before reading $|w_j|$ as importance.

## 7. Practical exercise: a comprehensive pipeline on California Housing

The book closes the chapter with an exercise on the **California Housing** dataset (bundled with scikit-learn; 20 640 districts, 8 numeric features, target = median house value in units of \$100 000). The task:

1. Load the data. 2. Split it. 3. Build a pipeline with **at least three steps**, ending in an estimator. 4. Fit it. 5. Evaluate on the test set.

> **📝 Note.** The book says "9 features", which counts the target column; the feature matrix itself has **8** columns (`MedInc, HouseAge, AveRooms, AveBedrms, Population, AveOccup, Latitude, Longitude`).

> ⚠️ **Run-it-yourself section.** `fetch_california_housing()` downloads the data from the internet on first use (then caches it locally). It could not be downloaded in the environment where this notebook was generated, so **the cells in this section have no stored output**. The code itself was checked end-to-end on a stand-in dataset with the same column names. Just run the cells below once with internet access and the results will appear.

I compare three candidate pipelines so the exercise becomes a small experiment:

| Pipeline | Steps |
|----------|-------|
| **A. Baseline** | scale → linear regression |
| **B. Poly + selection** | scale → polynomial features (degree 2) → `SelectFromModel(Lasso)` → ridge regression |
| **C. Trees** | impute → `SelectFromModel(random forest)` → histogram gradient boosting |

In [34]:
from sklearn.datasets import fetch_california_housing

housing = fetch_california_housing(as_frame=True)
X_cal, y_cal = housing.data, housing.target

print("features:", X_cal.shape, " target:", y_cal.shape)
X_cal.describe().T[["mean", "std", "min", "max"]].round(2)

features: (20640, 8)  target: (20640,)


,mean,std,min,max
MedInc,3.87,1.90,0.50,15.00
HouseAge,28.64,12.59,1.00,52.00
AveRooms,5.43,2.47,0.85,141.91
AveBedrms,1.10,0.47,0.33,34.07
Population,1425.48,1132.46,3.00,35682.00
AveOccup,3.07,10.39,0.69,1243.33
Latitude,35.63,2.14,32.54,41.95
Longitude,-119.57,2.00,-124.35,-114.31


In [35]:
X_cal_train, X_cal_test, y_cal_train, y_cal_test = train_test_split(
    X_cal, y_cal, test_size=0.2, random_state=42
)
print("train:", X_cal_train.shape, " test:", X_cal_test.shape)

train: (16512, 8)  test: (4128, 8)


In [36]:
from sklearn.linear_model import Lasso
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor

candidates = {
    "A. Baseline (scale -> linear)": Pipeline([
        ("scaler", StandardScaler()),
        ("model",  LinearRegression()),
    ]),
    "B. Poly + selection + ridge": Pipeline([
        ("scaler", StandardScaler()),
        ("poly",   PolynomialFeatures(degree=2, include_bias=False)),
        ("select", SelectFromModel(Lasso(alpha=0.01, max_iter=10000))),
        ("model",  Ridge(alpha=1.0)),
    ]),
    "C. Trees (impute -> select -> HGB)": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("select",  SelectFromModel(RandomForestRegressor(n_estimators=50, random_state=0, n_jobs=-1))),
        ("model",   HistGradientBoostingRegressor(random_state=0)),
    ]),
}

In [37]:
from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error

rows = {}
for name, pipe in candidates.items():
    pipe.fit(X_cal_train, y_cal_train)                    # one call: all steps
    pred = pipe.predict(X_cal_test)                       # one call: all steps again, on unseen data
    rows[name] = {"R^2":  r2_score(y_cal_test, pred),
                  "RMSE": root_mean_squared_error(y_cal_test, pred),
                  "MAE":  mean_absolute_error(y_cal_test, pred)}

pd.DataFrame(rows).T.sort_values("R^2", ascending=False).round(3)

,R^2,RMSE,MAE
C. Trees (impute -> select -> HGB),0.578,0.744,0.549
A. Baseline (scale -> linear),0.576,0.746,0.533
B. Poly + selection + ridge,0.501,0.808,0.492


How to read the results: **R²** is the share of variance explained (higher is better); **RMSE** and **MAE** are errors in the target's units, i.e. multiples of \$100 000 (lower is better). RMSE penalises large errors more than MAE.

Things worth examining once the table appears:

* Does adding polynomial features (B) beat the plain linear baseline (A)? Housing prices depend non-linearly on location and income, so a gain is expected.
* Does the tree-based pipeline (C) win without any scaling? Boosted trees usually handle this kind of tabular data well.
* Which features survived selection? The next cell shows it for pipeline B.

In [38]:
pipe_b = candidates["B. Poly + selection + ridge"]
selected = pipe_b[:-1].get_feature_names_out()            # [:-1] = all steps except the final model

print(f"{len(selected)} of {pipe_b.named_steps['poly'].n_output_features_} polynomial features were kept:")
print(selected.tolist())

27 of 44 polynomial features were kept:
['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'AveOccup', 'Latitude', 'Longitude', 'MedInc^2', 'MedInc HouseAge', 'MedInc AveRooms', 'MedInc AveBedrms', 'MedInc Population', 'MedInc Latitude', 'MedInc Longitude', 'HouseAge^2', 'HouseAge Population', 'HouseAge Latitude', 'HouseAge Longitude', 'AveRooms^2', 'AveRooms AveBedrms', 'AveRooms AveOccup', 'AveRooms Latitude', 'Population^2', 'Population Latitude', 'AveOccup^2', 'AveOccup Latitude', 'Latitude^2']


## 8. Decision guide: which transformation for which data?

The book condenses the scaling choices into a flow chart (its Figure 2.17). Re-expressed as a table:

**Step 1: are there outliers?**

| Answer | Prefer |
|--------|--------|
| Yes | `RobustScaler`, `QuantileTransformer` or `PowerTransformer` |
| No | continue to Step 2 |

**Step 2: what does the distribution look like?**

| Distribution | Prefer |
|--------------|--------|
| Roughly normal | `StandardScaler` or `MinMaxScaler` |
| Skewed | `PowerTransformer` (Yeo-Johnson, or Box-Cox for strictly positive data) |
| Unknown / mixed | `QuantileTransformer` or `Normalizer` |
| Sparse | `MaxAbsScaler` or `Normalizer` |

**Step 3: any special requirement?**

| Requirement | Use |
|-------------|-----|
| Values must lie in [0, 1] | `MinMaxScaler` |
| Each sample must have unit length | `Normalizer` |
| Zeros must stay zeros (keeps sparsity) | `MaxAbsScaler` |
| A custom range, e.g. [-1, 1] | `MinMaxScaler(feature_range=(-1, 1))` |

Two reminders that go with the chart: choose the scaler together with the **model** (trees need none; k-NN/SVM/linear models usually do), and always **fit it on the training data only**.

## Chapter summary

| Topic | Tools | One-line takeaway |
|-------|-------|-------------------|
| Raw-data problems | missing values, outliers, text categories, scale, leakage | check for each before modelling |
| Missing data | `SimpleImputer`, `KNNImputer`, `IterativeImputer` | cheap by default; smarter imputers help only when columns are related |
| Scaling | `StandardScaler`, `MinMaxScaler`, `Normalizer`, `RobustScaler`, ... | essential for k-NN/SVM/linear/NN models; pointless for trees |
| Encoding | `OneHotEncoder`, `OrdinalEncoder`, `LabelEncoder`, `ColumnTransformer` | one-hot for nominal, ordinal with explicit order for ordered, label encoding for targets |
| Pipelines | `Pipeline`, `make_pipeline`, `set_config(display="diagram")` | split first, put every data-dependent step inside |
| Feature creation | `PolynomialFeatures`, `KBinsDiscretizer` | new columns can give a simple model the power to fit curves |
| Feature selection | `RFE`, `RFECV`, `SelectFromModel` | RFE is thorough but slower; `SelectFromModel` is fast but blunt |

### Key takeaways

* **Garbage in, garbage out**, and preprocessing is continuous work, not a one-time cleanup.
* **Fit on training data only, then transform everything else.** A pipeline enforces this and prevents leakage (our noise experiment turned 50 % into a fake 84 %).
* **The right tool depends on the data *and* the model**: relationships between columns decide which imputer pays off; model type decides whether scaling matters.
* **Know your variable type** before encoding: nominal → one-hot, ordinal → `OrdinalEncoder`, target → `LabelEncoder`.
* **Feature engineering** is about giving the model better raw material, whether by creating features or by removing noisy ones.

### Self-check questions

<details><summary><b>Q1.</b> What is the difference between standardization, min-max scaling and <code>Normalizer</code>?</summary>

Standardization and min-max scaling work per <i>column</i> (mean 0 / std 1, or range [0, 1]). <code>Normalizer</code> works per <i>row</i>, rescaling each sample to unit norm (L1 or L2).
</details>

<details><summary><b>Q2.</b> Why must the train/test split come <i>before</i> preprocessing?</summary>

Statistics computed on all rows (means, ranges, selected features) leak information about the test set into training, which inflates evaluation scores. Fitting on the training set only and merely transforming the test set simulates truly unseen data (see the noise experiment in Section 5.3).
</details>

<details><summary><b>Q3.</b> When does <code>KNNImputer</code> or <code>IterativeImputer</code> beat <code>SimpleImputer</code>, and when not?</summary>

When features are correlated, since they borrow information from other columns (error cut by roughly 40-50 % in Section 2.5). When features are independent they offer no gain over the mean while costing more.
</details>

<details><summary><b>Q4.</b> Why is <code>LabelEncoder</code> a poor choice for an input feature like Position?</summary>

It numbers categories alphabetically, creating an order and distances that may not exist (Manager coded below Senior). Use <code>OneHotEncoder</code> for nominal features or <code>OrdinalEncoder</code> with an explicit order for ordinal ones.
</details>

<details><summary><b>Q5.</b> Does a random forest need scaled features?</summary>

No. Trees only compare a value with a threshold, and monotonic rescaling does not change the ordering, so the splits are the same (identical scores in Section 3.2). Recent scikit-learn versions also let forests accept missing values directly.
</details>

<details><summary><b>Q6.</b> What is the difference between <code>RFE</code> and <code>SelectFromModel</code>?</summary>

<code>RFE</code> retrains the model repeatedly, dropping the weakest feature each round until the desired number remains. <code>SelectFromModel</code> trains once and keeps features whose importance exceeds a threshold: faster, but the cut-off is cruder.
</details>

---
**Reference:** Sukup, J. *scikit-learn Cookbook, Third Edition*, Chapter 2, "Pre-Model Workflow and Data Preprocessing". Examples were re-run and extended by the author of this notebook; explanations are paraphrased and supplemented with extra experiments.